# 판정·합치기 평가 (담당: 홍예원, B)

- #12 판정 3종 비교 (kNN / 중심점 / 로지스틱 회귀) — 검증셋
- #14 온도 보정, #15 가중합·가중곱·RRF 비교 — 검증셋
- #26 최종 평가 (Top-1·Top-3, ECE, 신뢰도 다이어그램) — **테스트셋은 여기서 한 번만**

## #9 kNN 투표 방식 — 검증셋 평가

`_knn_logits`의 kNN 가중 투표 로짓을 **검증셋(`split == "val"`)** 으로 `k = 5, 10, 20`의
Top-1·Top-3 정확도로 본다. `by="artist"`와 `by="style"`를 모두 계산하되 비교 표의 기준은
artist로 두고 style은 참고로 함께 본다. k가 작으면 표를 받은 이름이 3개 미만일 수 있어
k별 "표를 받은 이름 수 평균"도 함께 본다. **테스트셋은 열지 않는다(NFR2).** 가짜 색인이면
수치를 기록하지 않고, 셀 출력은 지운 채로 둔다.

In [ ]:
import inspect
import pathlib
import sys

import numpy as np
import pandas as pd

# 노트북을 notebooks/ 폴더에서 실행해도 modules를 임포트할 수 있게 저장소 루트를 sys.path에 넣는다.
_root = pathlib.Path.cwd()
if not (_root / "modules").is_dir() and (_root.parent / "modules").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from modules import classify, data

# 색인을 불러오고 가짜 색인 여부와 화가 수를 확인한다.
index = data.load_index()
meta = index["meta"]

is_fake = index["meta"].attrs.get("fake")
print("fake 색인:", is_fake)
if is_fake:
    print("가짜 색인 — 수치 기록 금지")

# 화가 수는 숫자로 고정하지 않고 색인에서 센다.
print("화가 수:", meta["artist"].nunique())

In [ ]:
# 검증셋(split=="val")으로 k=5,10,20의 Top-1·Top-3를 by="artist"·"style" 둘 다 계산한다.
# 테스트셋은 열지 않는다(NFR2).
emb = index["emb"]
val_mask = (meta["split"] == "val").to_numpy()
E_val = emb[val_mask]

# eps는 하드코딩하지 않고 _knn_logits 시그니처의 기본값에서 읽는다(기본값이 바뀌어도 따라감).
_eps = inspect.signature(classify._knn_logits).parameters["eps"].default

ks_neighbors = (5, 10, 20)
rows = []
for by in ("artist", "style"):
    labels = meta[by].to_numpy()[val_mask].tolist()
    for k in ks_neighbors:
        names, z = classify._knn_logits(E_val, k=k, by=by)
        acc = classify._topk_accuracy(z, names, labels, ks=(1, 3))
        # 표를 받은 이름 수의 평균: z > log(_eps)인 이름(투표 > 0) 수를 입력별로 세어 평균낸다.
        n_voted = (z > np.log(_eps) + 1e-9).sum(axis=1).mean()
        rows.append(
            {
                "by": by,
                "k": k,
                "Top-1": acc[1],
                "Top-3": acc[3],
                "표받은_이름수_평균": float(n_voted),
            }
        )

result = pd.DataFrame(rows)

In [ ]:
# 비교 표의 기준은 artist, style은 참고로 함께 본다(FR4.1/FR4.4).
artist_table = result[result["by"] == "artist"].set_index("k")[
    ["Top-1", "Top-3", "표받은_이름수_평균"]
]
style_table = result[result["by"] == "style"].set_index("k")[
    ["Top-1", "Top-3", "표받은_이름수_평균"]
]
print("[기준] artist — k별 Top-1·Top-3·표받은 이름 수 평균")
print(artist_table)
print("\n[참고] style")
print(style_table)
# 가짜 색인이면 위 수치는 기록하지 않는다(NFR2). 셀 출력은 지운 채로 커밋한다(FR4.5).